# Multi-Head Attention

单头 Attention 的流程是：

```
X → (W_Q, W_K, W_V) → Q, K, V → Attention → 输出
```

Multi-Head Attention 只多了三步：

```
X → (W_Q, W_K, W_V) → Q, K, V
  → 拆分成 h 个头（reshape）
  → 每个头独立跑 Attention
  → 拼接（concat）
  → 乘输出投影矩阵 W_O
  → 最终输出
```

就这四个新动作：**拆分 → 并行计算 → 拼接 → 输出投影**。

# 1.线性层映射，生成Q、K、V

In [8]:
import numpy as np 

# 设置超参数
L = 7  # 序列长度
d_model = 8 # 词嵌入维度
h = 2  # 注意力头数(实际常用为8或16)
d_k = d_model // h  # 每个注意力头的维度 = 8 // 2 = 4

np.random.seed(42) # 设置随机种子以获得可重复的结果

# 模拟 embedding 后的输入 X
X = np.random.randn(L, d_model) # (7, 8)
print(f"输入矩阵shape：{X.shape}")

# 三个权重矩阵（实际中是可学习参数）
W_Q = np.random.randn(d_model, d_model)
W_K = np.random.randn(d_model, d_model)
W_V = np.random.randn(d_model, d_model)

# print(f"W_Q:\n{W_Q}")
# print(f"W_K:\n{W_K}")
# print(f"W_V:\n{W_V}")

# 线性映射，得到 Q、K、V
Q = X @ W_Q  # (7, 8) @ (8, 8) = (7, 8)
K = X @ W_K  # (7, 8) @ (8, 8) = (7, 8)
V = X @ W_V  # (7, 8) @ (8, 8) = (7, 8)

print(f"Q: {Q.shape}")
print(f"K: {K.shape}")
print(f"V: {V.shape}")


输入矩阵shape：(7, 8)
Q: (7, 8)
K: (7, 8)
V: (7, 8)


# 2.Head 拆分——把 d_model 切成 h 份

In [9]:
# reshape: (L, d_model) -> (L, h, d_k) -> (h, L, d_k)

Q_split = Q.reshape(L, h, d_k)
K_split = K.reshape(L, h, d_k)
V_split = V.reshape(L, h, d_k)

print(f"\n拆分后的Q、K、V矩阵shape：")
print(f"Q_split: {Q_split.shape}")
print(f"K_split: {K_split.shape}")
print(f"V_split: {V_split.shape}")

# 转置为 （h, L, d_k），方便每个注意力头独立计算
Q_head = Q_split.transpose(1,0,2)
K_head = K_split.transpose(1,0,2)
V_head = V_split.transpose(1,0,2)

print(f"\n转置后的Q、K、V矩阵shape：")
print(f"Q_head: {Q_head.shape}")
print(f"K_head: {K_head.shape}")
print(f"V_head: {V_head.shape}")


拆分后的Q、K、V矩阵shape：
Q_split: (7, 2, 4)
K_split: (7, 2, 4)
V_split: (7, 2, 4)

转置后的Q、K、V矩阵shape：
Q_head: (2, 7, 4)
K_head: (2, 7, 4)
V_head: (2, 7, 4)


# 3.每个头独立跑 Attention
拆分好之后，每个头的计算和单头完全一样：QKᵀ → 缩放 → Softmax → 加权 V。

In [11]:
def softmax(x):
    e = np.exp(x - np.max(x,axis=-1, keepdims=True))
    return e / e.sum(axis=-1,keepdims=True)

def single_head_attention(Q, K, V):
    """单个头的 Attention，输入输出都是 (L, d_k)"""
    d_k =Q.shape[-1]
    scores = Q@K.T
    scores /= np.sqrt(d_k)
    weights = softmax(scores)
    return weights @ V

# 对每个头分别计算
head_outputs = []
for i in range(h):
    out_i = single_head_attention(Q_head[i], K_head[i], V_head[i])
    head_outputs.append(out_i)
    print(f"Head {i} 输出：{out_i.shape}")

Head 0 输出：(7, 4)
Head 1 输出：(7, 4)


# 4.Concat 拼接——把多个头合并回来
2 个头算完了，怎么合并？**横向拼接**，沿着最后一个维度 concat。

In [15]:
# 先把 list 转成数组（h, L, d_k)
head_outputs = np.stack(head_outputs, axis=0)
print(f"\n拼接前（stack):{head_outputs.shape}")

# 转置回(L, h, d_k)，再 reshape 回 (L, d_model)
head_outputs = head_outputs.transpose(1,0,2)
print(f"转置后:{head_outputs.shape}")
# reshape 回 (L, d_model)
output = head_outputs.reshape(L, d_model)
print(f"最终输出矩阵shape：{output.shape}")


拼接前（stack):(2, 7, 4)
转置后:(7, 2, 4)
最终输出矩阵shape：(7, 8)


# 5.输出投影 W_O——让多个头"融合对话"
拼接之后，还差最后一步：乘输出投影矩阵 $W_O$。

In [16]:
# 输出投影矩阵W_O: (d_model, d_model)
W_O = np.random.randn(d_model, d_model)

# 最终输出矩阵
final_output = output @ W_O
print(f"\n输出投影后：{final_output.shape}")
print(f"\n输入 X 的形状：{X.shape}")
print(f"形状是否一致：{final_output.shape == X.shape}")


输出投影后：(7, 8)

输入 X 的形状：(7, 8)
形状是否一致：True


# 完整代码：把五步打包成一个函数

In [17]:
import numpy as np

def multi_head_attention(X, W_Q, W_K, W_V, W_O, h):
    """
    Multi-Head Attention 完整实现
    
    参数:
        X:    输入矩阵, shape (L, d_model)
        W_Q, W_K, W_V: 线性投影矩阵, shape (d_model, d_model)
        W_O:  输出投影矩阵, shape (d_model, d_model)
        h:    注意力头数
    
    返回:
        output: shape (L, d_model)
    """
    L, d_model = X.shape
    d_k = d_model // h

    # ① 线性映射
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    print(f"[①线性映射] Q/K/V: {Q.shape}")

    # ② 拆分成 h 个头: (L, d_model) → (h, L, d_k)
    def split_heads(M):
        return M.reshape(L, h, d_k).transpose(1, 0, 2)

    Q_h = split_heads(Q)
    K_h = split_heads(K)
    V_h = split_heads(V)
    print(f"[②Head拆分] Q_h/K_h/V_h: {Q_h.shape}")

    # ③ 每个头独立计算 Attention
    def softmax(x):
        e = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return e / e.sum(axis=-1, keepdims=True)

    head_outs = []
    for i in range(h):
        scores = Q_h[i] @ K_h[i].T / np.sqrt(d_k)
        attn = softmax(scores) @ V_h[i]
        head_outs.append(attn)
    print(f"[③并行Attention] 每个head输出: {head_outs[0].shape}")

    # ④ Concat 拼接: (h, L, d_k) → (L, d_model)
    concat = np.stack(head_outs, axis=0).transpose(1, 0, 2).reshape(L, d_model)
    print(f"[④Concat拼接] 拼接后: {concat.shape}")

    # ⑤ 输出投影
    output = concat @ W_O
    print(f"[⑤输出投影] 最终输出: {output.shape}")

    return output


# ——— 运行测试 ———
if __name__ == "__main__":
    np.random.seed(42)
    L, d_model, h = 7, 8, 2

    X   = np.random.randn(L, d_model)
    W_Q = np.random.randn(d_model, d_model)
    W_K = np.random.randn(d_model, d_model)
    W_V = np.random.randn(d_model, d_model)
    W_O = np.random.randn(d_model, d_model)

    print("=== Multi-Head Attention ===")
    out = multi_head_attention(X, W_Q, W_K, W_V, W_O, h)
    print(f"\n输入形状:  {X.shape}")
    print(f"输出形状:  {out.shape}")
    print(f"形状一致:  {X.shape == out.shape}")

=== Multi-Head Attention ===
[①线性映射] Q/K/V: (7, 8)
[②Head拆分] Q_h/K_h/V_h: (2, 7, 4)
[③并行Attention] 每个head输出: (7, 4)
[④Concat拼接] 拼接后: (7, 8)
[⑤输出投影] 最终输出: (7, 8)

输入形状:  (7, 8)
输出形状:  (7, 8)
形状一致:  True
